In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pickle
import gzip
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def unpickle(filename):
    file = gzip.open(filename, 'rb')
    data = pickle.load(file, encoding='latin1')
    file.close()
    return data

In [ ]:
dir = '/content/drive/MyDrive/honors_thesis_data/20_nodes_is_connected_spring_layout/'
#dir2 = '/content/drive/MyDrive/honors_thesis_data/20_nodes_is_planar_random_layout/'

train_inputs = []
train_results = []
test_inputs = []
test_results = []
for i in range(6):
  data = unpickle(dir + 'training' + str(i+1) + '.pkl.gz')
  for j in range(len(data[0])):
    train_inputs.append(data[0][j])
    train_results.append(data[1][j])

data = unpickle(dir + 'testing.pkl.gz')
for j in range(len(data[0])):
  test_inputs.append(data[0][j])
  test_results.append(data[1][j])
data.clear()

In [ ]:
class MyDataset(Dataset):
    def __init__(self, inputs, results, transform=None):
        self.inputs = inputs
        self.results = results
        self.transform = transform

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        x = self.inputs[idx]
        y = self.results[idx]
        # Convert numpy array to PIL Image before applying transforms
        x = Image.fromarray((x * 255).astype('uint8'))
        if self.transform:
            x = self.transform(x)
        return x, y

In [ ]:
transform = transforms.Compose([
    #transforms.Grayscale(),          # remove if already grayscale
    transforms.Resize((28, 28)),     # match MNIST size
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_without_resize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
train_dataset = MyDataset(train_inputs, train_results, transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
train_orig_dataset = MyDataset(train_inputs, train_results, transform=transform_without_resize)

test_dataset = MyDataset(test_inputs, test_results, transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
test_orig_dataset = MyDataset(test_inputs, test_results, transform=transform_without_resize)

In [ ]:
def save_model(net, filename):
  with gzip.open(filename, 'wb') as f:
    pickle.dump(net, f)

def load_model(filename):
  with gzip.open(filename, 'rb') as f:
    return pickle.load(f)

#One neuron output layer

In [ ]:
class BinaryModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        #self.fc1 = nn.Linear(56*56, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)  # ONE output

    def forward(self, x):
        x = x.view(-1, 28*28)
        #x = x.view(-1, 56*56)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # NO sigmoid here

In [ ]:
model = BinaryModel()
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train(model, loader):
    model.train()
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        labels = labels.float().unsqueeze(1)  # shape [B, 1]
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            probs = torch.sigmoid(outputs)      # convert logits → probabilities
            preds = (probs > 0.5).int().squeeze()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            labels = labels.float().unsqueeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / total
    accuracy = 100 * correct / total
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Average Loss: {avg_loss:.4f}")

In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_one_neuron_model.pkl.gz')

In [ ]:
model = load_model(dir2 + 'torch_one_neuron_model.pkl.gz')
model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
evaluate(model, train_loader, criterion)

In [ ]:
def get_images(model, loader, prediction, num_images=5, is_test_data=True):
    model.eval()
    correct_images = []
    wrong_images = []
    TP = 0
    TN = 0
    FP = 0
    FN = 0
    i = -1
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int().squeeze()
            for image, label, pred, output in zip(images, labels, preds, probs):
              i += 1
              output = output.item()
              if pred == label:
                  if pred == 1:
                      TP += 1
                  else:
                      TN += 1
                  if len(correct_images) == num_images:
                      continue
                  if pred == prediction:
                    if is_test_data:
                      correct_images.append((image, test_orig_dataset[i][0], output))
                    else:
                      correct_images.append((image, train_orig_dataset[i][0], output))
              else:
                  if pred == 1:
                      FP += 1
                  else:
                      FN += 1
                  if len(wrong_images) == num_images:
                      continue
                  if pred == prediction:
                    if is_test_data:
                      wrong_images.append((image, test_orig_dataset[i][0], output))
                    else:
                      wrong_images.append((image, train_orig_dataset[i][0], output))
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}\n")
    return correct_images, wrong_images

In [ ]:
model = load_model(dir + 'torch_one_neuron_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output) in correct:
  print(f"{output:.4f}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output) in wrong:
  print(f"{output:.4f}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output) in correct:
  print(f"{output:.4f}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output) in wrong:
  print(f"{output:.4f}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Two neurons output layer

In [ ]:
class BinaryCEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
model = BinaryCEModel()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train(model, loader):
  model.train()
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)     # shape: [B, 2]
    loss = criterion(outputs, labels)  # labels: [B]

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    preds = torch.argmax(outputs, dim=1)
    correct = (preds == labels).sum().item()


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Record metrics
            total_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = (total_correct / total) * 100

    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Average Loss: {avg_loss:.4f}")


In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_two_neuron_model.pkl.gz')

In [ ]:
model = load_model(dir2 + 'torch_two_neuron_model.pkl.gz')
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
evaluate(model, train_loader, criterion)

In [ ]:
def get_images(model, loader, prediction, num_images=5, is_test_data=True):
    model.eval()
    correct_images = []
    wrong_images = []
    TP = 0
    TN = 0
    FP = 0
    FN = 0
    i = -1
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            for image, label, prob, pred in zip(images, labels, probs, preds):
              i += 1
              prob = prob.tolist()
              pred = pred.item()
              certainty = prob[pred]
              if pred == label:
                  if pred == 1:
                      TP += 1
                  else:
                      TN += 1
                  if len(correct_images) == num_images:
                      continue
                  if pred == prediction:
                    if is_test_data:
                      correct_images.append((image, test_orig_dataset[i][0], certainty, pred))
                    else:
                      correct_images.append((image, train_orig_dataset[i][0], certainty, pred))
              else:
                  if pred == 1:
                      FP += 1
                  else:
                      FN += 1
                  if len(wrong_images) == num_images:
                      continue
                  if pred == prediction:
                    if is_test_data:
                      wrong_images.append((image, test_orig_dataset[i][0], certainty, pred))
                    else:
                      wrong_images.append((image, train_orig_dataset[i][0], certainty, pred))
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}\n")
    return correct_images, wrong_images


In [ ]:
model = load_model(dir + 'torch_two_neuron_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=7)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[3:]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[0:5]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=7)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[3:]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[0:5]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Convolutional layers

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: 1×28×28 (grayscale)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # After two pools: 32 × 7 × 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)  # 2 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 16×14×14
        x = self.pool(F.relu(self.conv2(x)))  # 32×7×7

        #x = x.view(x.size(0), -1)
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # logits

In [ ]:
model = SimpleCNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_cnn_model.pkl.gz')

In [ ]:
model = load_model(dir + 'torch_cnn_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=12)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[7:12]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[5:10]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=12)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[7:12]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[5:10]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Data augmentation

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.RandomRotation(10, fill=255),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
        fill=255
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
train_aug_dataset = MyDataset(train_inputs, train_results, train_transform)
train_aug_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_aug_dataset = MyDataset(test_inputs, test_results, test_transform)
test_aug_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
img, _ = train_aug_dataset[0]
plt.imshow(img.squeeze(), cmap="gray")
plt.axis("off")


In [ ]:
model = SimpleCNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_aug_loader)

  evaluate(model, train_aug_loader, criterion)
  evaluate(model, test_aug_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_cnn_data_aug_model.pkl.gz')

In [ ]:
model = load_model(dir + 'torch_cnn_data_aug_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=15)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[10:15]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[5:10]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=15)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[10:15]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[5:10]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Regularization

#Weight decay (L2 regularization)

In [ ]:
model = SimpleCNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)


In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_cnn_weight_decay_model.pkl.gz')

In [ ]:
model = load_model(dir + 'torch_cnn_weight_decay_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=17)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[12:17]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[10:15]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=17)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[12:17]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[10:15]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Dropout

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [ ]:
model = CNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(3):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_cnn_dropout_model.pkl.gz')

In [ ]:
model = load_model(dir + 'torch_cnn_dropout_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=22)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[17:22]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[15:20]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=22)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[17:22]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[15:20]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Batch Normalization

In [ ]:
class SimpleCNNWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()

        # Conv1: bias=False because BN1 follows
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # Conv2: bias=False because BN2 follows
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(32)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        #self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        # Order: Conv -> BatchNorm -> ReLU -> Pool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
model = SimpleCNNWithBatchNorm()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(1):
  print(f"Epoch {epoch+1}")
  train(model, train_loader)

  evaluate(model, train_loader, criterion)
  evaluate(model, test_loader, criterion)

In [ ]:
save_model(model, dir + 'torch_cnn_batch_norm_model.pkl.gz')

In [ ]:
model = load_model(dir2 + 'torch_cnn_batch_norm_model.pkl.gz')
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
evaluate(model, train_loader, criterion)

In [ ]:
model = load_model(dir + 'torch_cnn_batch_norm_model.pkl.gz')
model.to(device)

In [ ]:
correct, wrong = get_images(model, test_loader, 1, num_images=27)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[22:27]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[17:22]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
correct, wrong = get_images(model, test_loader, 0, num_images=27)
print(len(correct), len(wrong))

In [ ]:
for (img, orig_img, output, pred) in correct[22:27]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

In [ ]:
for (img, orig_img, output, pred) in wrong[17:22]:
  print(f"{output:.4f} Prediction: {pred}")
  plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
  plt.axis("off")
  plt.show()
  plt.imshow(orig_img.squeeze(), cmap="gray")
  plt.axis("off")
  plt.show()

#Image resizing

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load image
img = Image.open("graph.png")

# Resize (width, height)
img_resized = img.resize((128, 128), Image.BILINEAR)

# Show image
plt.imshow(img_resized)
plt.axis("off")
plt.show()


#Integrated Gradiant

In [ ]:
def integrated_gradients(
    model,
    x,
    target_class,
    baseline=None,
    steps=50
):
    model.eval()

    if baseline is None:
        baseline = torch.zeros_like(x)

    # Scale inputs
    alphas = torch.linspace(0, 1, steps).to(x.device)

    ig = torch.zeros_like(x)

    for alpha in alphas:
        x_interp = baseline + alpha * (x - baseline)
        x_interp.requires_grad_(True)

        model.zero_grad()

        output = model(x_interp)

        # Select target class logit
        target_logit = output[:, target_class]

        target_logit.backward(torch.ones_like(target_logit))

        ig += x_interp.grad.detach()

    ig = ig / steps
    ig = (x - baseline) * ig

    return ig


In [ ]:
model = load_model(dir + 'torch_cnn_weight_decay_model.pkl.gz')
model.to(device)

In [ ]:
x = torch.randn(1, 1, 28, 28)
x = x.to(device)
y = model(x)
print(y.shape)

In [ ]:
x = x.squeeze()           # remove extra dimensions if any
x = x.unsqueeze(0)        # batch dimension
x = x.unsqueeze(0)        # channel dimension if missing
print(x.shape)            # should be [1, 1, 28, 28]

In [ ]:
with torch.no_grad():
    logit = model(x)
    if (logit.shape[1] == 1):
        pred = (logit > 0).long().item()
    else:
        pred = logit.argmax(dim=1).item()

In [ ]:
x, y = train_dataset[5003]
x2, y2 = train_orig_dataset[5003]
x = x.unsqueeze(0)
x = x.to(device)

# Plot original image
plt.imshow(x.squeeze().cpu().numpy(), cmap="gray")
plt.show()
plt.imshow(x2.squeeze().cpu().numpy(), cmap="gray")
#plt.axis("off")
plt.show()


In [ ]:
ig = integrated_gradients(
    model,
    x,
    target_class=pred,
    steps=50
)

In [ ]:
print(y)
plt.imshow(x.squeeze().cpu().numpy(), cmap="gray")
plt.show()

attr = ig.squeeze().detach().cpu().numpy()
plt.imshow(attr, cmap="hot")
plt.colorbar()
plt.show()

#Binary Integrated Gradiant

In [ ]:
def integrated_gradients(
    model,
    x,
    baseline=None,
    steps=50
):
    model.eval()

    if baseline is None:
        # TRUE black in normalized space
        baseline = torch.full_like(x, -1.0)

    # Generate scaled inputs in one batch (vectorized)
    alphas = torch.linspace(0, 1, steps, device=x.device).view(-1, 1, 1, 1)
    interpolated = baseline + alphas * (x - baseline)
    interpolated.requires_grad_(True)

    # Forward pass
    outputs = model(interpolated).squeeze()

    # Since binary model has ONE logit:
    grads = torch.autograd.grad(
        outputs=outputs,
        inputs=interpolated,
        grad_outputs=torch.ones_like(outputs),
        create_graph=False,
        retain_graph=False
    )[0]

    # Average gradients across steps
    avg_grads = grads.mean(dim=0, keepdim=True)

    # IG formula
    ig = (x - baseline) * avg_grads

    return ig


In [ ]:
model = load_model(dir + 'torch_one_neuron_model.pkl.gz')
model.to(device)
model.eval()

# Get sample
x, y = train_dataset[0]
x = x.unsqueeze(0).to(device)

# Compute IG
ig = integrated_gradients(model, x, steps=50)

# Plot original image
plt.imshow(x.squeeze().cpu().numpy(), cmap="gray")
plt.axis("off")
plt.show()

# Plot attribution
attr = ig.squeeze().detach().cpu().numpy()

plt.imshow(attr, cmap="hot")
plt.colorbar()
plt.axis("off")
plt.show()


In [ ]:
def get_image_gradient(model, image):
    model.eval()

    image_tensor = image.unsqueeze(0).to(device)
    image_tensor.requires_grad_(True)

    output = model(image_tensor).squeeze()
    output.backward()

    gradient = image_tensor.grad.detach().squeeze(0)
    return gradient


In [ ]:
img, target = train_dataset[0]
grad = get_image_gradient(model, img)

attr = grad.squeeze().cpu().numpy()

plt.imshow(attr, cmap="hot")
plt.colorbar()
plt.axis("off")
plt.show()

plt.imshow(img.squeeze().cpu().numpy(), cmap="gray")
plt.axis("off")
plt.show()

#CNN Integrated Gradiant

In [ ]:
def integrated_gradients(
    model,
    x,
    target_class,
    baseline=None,
    steps=50
):
    model.eval()

    if baseline is None:
        # True black in your normalized space
        baseline = torch.full_like(x, -1.0)

    # Create interpolation batch
    alphas = torch.linspace(0, 1, steps, device=x.device).view(-1, 1, 1, 1)
    interpolated = baseline + alphas * (x - baseline)
    interpolated.requires_grad_(True)

    # Forward pass
    outputs = model(interpolated)

    # Select target class logit
    target_logits = outputs[:, target_class]

    # Compute gradients
    grads = torch.autograd.grad(
        outputs=target_logits,
        inputs=interpolated,
        grad_outputs=torch.ones_like(target_logits),
        create_graph=False,
        retain_graph=False
    )[0]

    # Average gradients
    avg_grads = grads.mean(dim=0, keepdim=True)

    # IG formula
    ig = (x - baseline) * avg_grads

    return ig


In [ ]:
model = load_model(dir + 'torch_cnn_model.pkl.gz')
model.to(device)
model.eval()

x, y = test_dataset[6001]
x = x.unsqueeze(0).to(device)

# Predict class
with torch.no_grad():
    pred = model(x).argmax(dim=1).item()

# Compute IG
ig = integrated_gradients(model, x, target_class=pred, steps=50)

In [ ]:
# Original image
plt.imshow(x.squeeze().cpu().numpy(), cmap="gray")
plt.show()

# Attribution
attr = ig.squeeze().detach().cpu().numpy()

plt.imshow(attr, cmap="hot")
plt.colorbar()
plt.show()